# Figure S2 — Development and benchmarking of CC-Perturb-seq

Second-pass implementation of benchmark panels b–n. Panel a remains the editable molecular-workflow schematic.


In [ ]:
import matplotlib.pyplot as plt
from src.cc_perturbseq.figure_notebook import get_path, initialize_figure, load_figure_manifest, panel_directory, source_asset_status, write_receipt
from src.cc_perturbseq.program_panels import *

def export(fig, panel, stem):
    out = panel_directory(ctx, panel)
    result = {}
    for suffix, kwargs in [("pdf", {}), ("svg", {}), ("png", {"dpi": 300})]:
        path = out / f"{stem}.{suffix}"
        fig.savefig(path, **kwargs)
        result[f"{stem}_{suffix}"] = str(path)
    return result

FIGURE = load_figure_manifest("Figure_S02")
ctx = initialize_figure(FIGURE)
ctx["input_audit"]


## Panel a — molecular workflow schematic


In [ ]:
source_asset_status(ctx, "a", "assets.s2a_ccperturbseq_schematic")


## Panels b–f — hiPSC circular capture versus CROP-seq


In [ ]:
hipsc_path = get_path(ctx["paths"], "benchmark.hipsc_pilot_anndata")
hipsc_down_path = get_path(ctx["paths"], "benchmark.hipsc_downsampling")
hipsc_kd_path = get_path(ctx["paths"], "benchmark.hipsc_knockdown")
hipsc_metrics = benchmark_cell_metrics(hipsc_path)
fig_bcd, detection = plot_capture_benchmark(hipsc_metrics)
downsampling = standardize_downsampling(hipsc_down_path)
fig_e = plot_downsampling(downsampling, outcome="transcript_umis", ylabel="Median transcriptome UMIs per cell")
knockdown = standardize_pilot_knockdown(hipsc_kd_path)
fig_f = plot_pilot_knockdown(knockdown)
outputs = {**export(fig_bcd, "b-f", "Figure_S02b_d_hipsc_capture_benchmark"), **export(fig_e, "b-f", "Figure_S02e_hipsc_saturation"), **export(fig_f, "b-f", "Figure_S02f_hipsc_knockdown")}
tables = save_source_tables({"cell_metrics": hipsc_metrics, "guide_detection": detection, "downsampling": downsampling, "knockdown": knockdown}, panel_directory(ctx, "b-f"), "Figure_S02b_f")
write_receipt(ctx, "b-f", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"anndata": str(hipsc_path), "downsampling": str(hipsc_down_path), "knockdown": str(hipsc_kd_path)}, outputs={**outputs, **tables}, parameters={"guide_metric": "top guide UMI"})
plt.show()


## Panels g–h — TeloHAEC capture benchmark


In [ ]:
telo_path = get_path(ctx["paths"], "benchmark.telohaec_pilot_anndata")
telo_metrics = benchmark_cell_metrics(telo_path)
fig_gh, telo_detection = plot_capture_benchmark(telo_metrics)
outputs = export(fig_gh, "g-h", "Figure_S02g_h_telohaec_capture_benchmark")
tables = save_source_tables({"cell_metrics": telo_metrics, "guide_detection": telo_detection}, panel_directory(ctx, "g-h"), "Figure_S02g_h")
write_receipt(ctx, "g-h", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"anndata": str(telo_path)}, outputs={**outputs, **tables}, parameters={"guide_metric": "top guide UMI"})
plt.show()


## Panels i–n — Parse Evercode v3 compatibility


In [ ]:
parse_path = get_path(ctx["paths"], "benchmark.parse_v3_anndata")
parse_guides = get_path(ctx["paths"], "benchmark.parse_v3_guide_counts", required=False)
parse_down_path = get_path(ctx["paths"], "benchmark.parse_v3_downsampling")
parse_metrics = standardize_parse_metrics(parse_path, parse_guides)
fig_parse = plot_parse_benchmark(parse_metrics)
parse_down = standardize_downsampling(parse_down_path)
fig_transcripts = plot_downsampling(parse_down, outcome="transcript_umis", ylabel="Median transcriptome UMIs per cell")
fig_genes = plot_downsampling(parse_down, outcome="detected_genes", ylabel="Median detected genes per cell")
outputs = {**export(fig_parse, "i-n", "Figure_S02i_n_parse_benchmark"), **export(fig_transcripts, "i-n", "Figure_S02m_parse_transcript_saturation"), **export(fig_genes, "i-n", "Figure_S02k_parse_gene_saturation")}
tables = save_source_tables({"cell_metrics": parse_metrics, "downsampling": parse_down}, panel_directory(ctx, "i-n"), "Figure_S02i_n")
write_receipt(ctx, "i-n", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"anndata": str(parse_path), "guide_counts": str(parse_guides), "downsampling": str(parse_down_path)}, outputs={**outputs, **tables}, parameters={"specificity": "matched-pool UMIs / total pool UMIs"})
plt.show()
